##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy? The fine-tuned ResNet50V2 achieved the highest accuracy, reaching 91.62%

- Which model trained faster? MobileNetV2 trained faster than ResNet50V2 because it is a lightweight architecture designed for efficiency.

- How might the architecture explain the differences? ResNet is deeper and can learn more complex features, which helps it achieve better accuracy. MobileNet focuses on efficiency, so it runs faster but usually gives slightly lower results.

In [1]:
# =============================
# 1) Import Libraries
# =============================
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [2]:
# =============================
# 2) Load CIFAR-10
# =============================
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

class_names = [
    "airplane","automobile","bird","cat","deer",
    "dog","frog","horse","ship","truck"
]

# convert labels to integers
y_train = y_train.squeeze().astype("int64")
y_test  = y_test.squeeze().astype("int64")

# convert images to float32
x_train = x_train.astype("float32")
x_test  = x_test.astype("float32")

print("Train shape:", x_train.shape)
print("Test shape :", x_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 70s 0us/step
Train shape: (50000, 32, 32, 3)
Test shape : (10000, 32, 32, 3)


In [3]:
# =============================
# 3) Data Augmentation
# =============================
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augmentation")

In [4]:
# =============================
# 4) Load MobileNetV2 Backbone
# =============================
mobilenet_base = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

mobilenet_base.trainable = False  # Freeze backbone first

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
# =============================
# 5) Build Full Model
# =============================
mobilenet_model = keras.Sequential([
    layers.Input(shape=(32,32,3)),
    data_augmentation,
    layers.Resizing(224,224),
    layers.Lambda(preprocess_input),   # IMPORTANT
    mobilenet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10)
], name="cifar10_mobilenetv2")

mobilenet_model.summary()

Model: "cifar10_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
# =============================
# 6) Compile Model (Frozen Backbone)
# =============================
mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1),
]

In [8]:
# =============================
# 7) Train (Feature Extraction)
# =============================
history_mb = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 78s 111ms/step - accuracy: 0.7679 - loss: 0.6671 - val_accuracy: 0.8282 - val_loss: 0.4969 - learning_rate: 0.0010
Epoch 2/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 78s 110ms/step - accuracy: 0.7716 - loss: 0.6552 - val_accuracy: 0.8446 - val_loss: 0.4572 - learning_rate: 0.0010
Epoch 3/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 78s 110ms/step - accuracy: 0.7723 - loss: 0.6606 - val_accuracy: 0.8358 - val_loss: 0.4857 - learning_rate: 0.0010
Epoch 4/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 78s 110ms/step - accuracy: 0.7868 - loss: 0.6157 - val_accuracy: 0.8384 - val_loss: 0.4767 - learning_rate: 5.0000e-04
Epoch 5/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 78s 111ms/step - accuracy: 0.7925 - loss: 0.5976 - val_accuracy: 0.8486 - val_loss: 0.4491 - learning_rate: 2.5000e-04
Epoch 6/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 78s 110ms/step - accuracy: 0.7916 - loss: 0.6013 - val_accuracy: 0.8412 - val_loss: 0.4622 - learning_rate: 2.5000e-04
Epoch 7/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 78s 111ms/step - a

In [9]:
# =============================
# 8) Evaluate Frozen Model
# =============================
test_loss_mb, test_acc_mb = mobilenet_model.evaluate(x_test, y_test, verbose=0)
print("MobileNetV2 (frozen) accuracy:", test_acc_mb)
print("MobileNetV2 (frozen) loss    :", test_loss_mb)

MobileNetV2 (frozen) accuracy: 0.8324000239372253
MobileNetV2 (frozen) loss    : 0.48205047845840454


In [10]:
# =============================
# 9) Fine-Tune Last Layers
# =============================
mobilenet_base.trainable = True

# Freeze early layers, train last ones
for layer in mobilenet_base.layers[:-20]:
    layer.trainable = False

print("Trainable layers:",
      sum(l.trainable for l in mobilenet_base.layers),
      "/", len(mobilenet_base.layers))

mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

Trainable layers: 20 / 154


In [11]:
# =============================
# 10) Train Fine-Tuned Model
# =============================
history_mb_ft = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 104s 135ms/step - accuracy: 0.6732 - loss: 0.9540 - val_accuracy: 0.8398 - val_loss: 0.4770
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 94s 133ms/step - accuracy: 0.7647 - loss: 0.6808 - val_accuracy: 0.8428 - val_loss: 0.4530
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 94s 133ms/step - accuracy: 0.7827 - loss: 0.6258 - val_accuracy: 0.8500 - val_loss: 0.4262


In [12]:
# =============================
# 11) Final Evaluation
# =============================
test_loss_ft, test_acc_ft = mobilenet_model.evaluate(x_test, y_test, verbose=0)

print("MobileNetV2 (fine-tuned) accuracy:", test_acc_ft)
print("MobileNetV2 (fine-tuned) loss    :", test_loss_ft)

MobileNetV2 (fine-tuned) accuracy: 0.8442000150680542
MobileNetV2 (fine-tuned) loss    : 0.4598940312862396
